# Notebook 20 — Constraint-Gated Macro Routing

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 19 compressed prototype memory into macro-prototypes.

Notebook 20 adds a CGCS-style constraint gate over compressed macro routes.

Constraint view:
> compressed memory is useful only when macro routes remain coherent, stable, and recoverable under observed pressure.

## Goals

1. Load Notebook 19 macro-routing outputs when available.
2. Compute CGCS-style macro-route scores.
3. Gate macro routes into:
   - accepted,
   - watch,
   - blocked / fallback.
4. Compare raw macro routing vs constraint-gated macro routing.
5. Identify routes that should expand / decompress.
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 19 compressed macro routing

In [ ]:
macro_path = RESULTS_DIR / "notebook19_recursive_memory_compression.csv"
compressed_path = RESULTS_DIR / "notebook19_compressed_macro_prototypes.csv"
proto_map_path = RESULTS_DIR / "notebook19_prototype_to_macro_map.csv"

if macro_path.exists():
    work = pd.read_csv(macro_path)
    print("Loaded:", macro_path)
else:
    work = None

if compressed_path.exists():
    compressed = pd.read_csv(compressed_path)
    print("Loaded:", compressed_path)
else:
    compressed = None

if proto_map_path.exists():
    proto_map = pd.read_csv(proto_map_path)
    print("Loaded:", proto_map_path)
else:
    proto_map = None

if work is None:
    print("Notebook 19 outputs not found; creating fallback macro-routing data.")
    rng = np.random.default_rng(42)
    n = 240
    macros = ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"]
    seq = []
    for i in range(n):
        if 95 <= i <= 145:
            seq.append("macro_0")
        else:
            seq.append(rng.choice(macros))
    work = pd.DataFrame({
        "window_id": np.arange(n),
        "macro_route": seq,
        "compressed_route_stability": np.clip(rng.normal(0.45, 0.18, n), 0, 1),
        "macro_switch_rate": np.clip(rng.normal(0.55, 0.25, n), 0, 1),
        "child_switch_rate": np.clip(rng.normal(0.62, 0.25, n), 0, 1),
        "policy_switch_rate": np.clip(rng.normal(0.58, 0.25, n), 0, 1),
        "macro_compression_residual": np.clip(rng.normal(0.12, 0.08, n), 0, 1),
        "compression_quality": np.clip(rng.normal(0.82, 0.10, n), 0, 1),
    })
    work.loc[95:145, "compressed_route_stability"] = np.clip(rng.normal(0.95, 0.05, 51), 0, 1)
    work.loc[95:145, "macro_switch_rate"] = 0.0

if compressed is None:
    compressed = pd.DataFrame({
        "macro_prototype": sorted(work["macro_route"].unique()),
        "member_count": [1] * work["macro_route"].nunique(),
        "mean_memory_stability": np.linspace(0.35, 0.75, work["macro_route"].nunique()),
        "mean_effective_memory_weight": np.linspace(0.45, 0.80, work["macro_route"].nunique()),
        "total_usage_count": [int((work["macro_route"] == m).sum()) for m in sorted(work["macro_route"].unique())],
    })

work.head(), compressed.head()

## Compute CGCS-style constraint components

Components:

- **coherence gate**: macro route stability / compression quality
- **pressure gate**: low switch pressure
- **reconstruction gate**: low macro residual
- **memory gate**: macro memory stability / effective weight

In [ ]:
def norm01(s, invert=False):
    s = pd.Series(s).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        vals = pd.Series(np.ones(len(s)) if not invert else np.zeros(len(s)), index=s.index)
    else:
        vals = (s - lo) / (hi - lo)
    if invert:
        vals = 1.0 - vals
    return vals.clip(0, 1)

work = work.copy()

for c in ["compressed_route_stability", "macro_switch_rate", "child_switch_rate", "policy_switch_rate",
          "macro_compression_residual", "compression_quality"]:
    if c not in work.columns:
        work[c] = 0.5
    work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0.5)

compressed = compressed.copy()
if "macro_prototype" not in compressed.columns:
    compressed["macro_prototype"] = sorted(work["macro_route"].unique())

for c in ["mean_memory_stability", "mean_effective_memory_weight", "member_count", "total_usage_count"]:
    if c not in compressed.columns:
        compressed[c] = 1.0
    compressed[c] = pd.to_numeric(compressed[c], errors="coerce").fillna(0.0)

macro_memory = compressed.set_index("macro_prototype")
work["macro_memory_stability"] = work["macro_route"].map(macro_memory["mean_memory_stability"]).fillna(0.5)
work["macro_effective_weight"] = work["macro_route"].map(macro_memory["mean_effective_memory_weight"]).fillna(0.5)
work["macro_member_count"] = work["macro_route"].map(macro_memory["member_count"]).fillna(1)

work["coherence_component"] = (
    0.60 * norm01(work["compressed_route_stability"]) +
    0.40 * norm01(work["compression_quality"])
).clip(0, 1)

work["pressure_component"] = (
    0.50 * norm01(work["macro_switch_rate"], invert=True) +
    0.25 * norm01(work["child_switch_rate"], invert=True) +
    0.25 * norm01(work["policy_switch_rate"], invert=True)
).clip(0, 1)

work["reconstruction_component"] = norm01(work["macro_compression_residual"], invert=True)

work["memory_component"] = (
    0.50 * norm01(work["macro_memory_stability"]) +
    0.50 * norm01(work["macro_effective_weight"])
).clip(0, 1)

work[["window_id", "macro_route", "coherence_component", "pressure_component", "reconstruction_component", "memory_component"]].head()

## Constraint-Gated Coherence Score

CGCS-style macro score:

```text
CGCS = 0.35 coherence + 0.25 pressure + 0.25 reconstruction + 0.15 memory
```

In [ ]:
work["macro_cgcs_score"] = (
    0.35 * work["coherence_component"] +
    0.25 * work["pressure_component"] +
    0.25 * work["reconstruction_component"] +
    0.15 * work["memory_component"]
).clip(0, 1)

accept_threshold = 0.70
watch_threshold = 0.50

def gate_label(x):
    if x >= accept_threshold:
        return "accepted"
    if x >= watch_threshold:
        return "watch"
    return "fallback"

work["constraint_gate"] = work["macro_cgcs_score"].apply(gate_label)
work["gated_macro_route"] = np.where(
    work["constraint_gate"].eq("fallback"),
    "fallback_route",
    work["macro_route"],
)

work["decompression_recommended"] = (
    (work["constraint_gate"].eq("fallback")) |
    ((work["constraint_gate"].eq("watch")) & (work["macro_member_count"] > 1)) |
    (work["macro_compression_residual"] > work["macro_compression_residual"].quantile(0.80))
)

work[["window_id", "macro_route", "macro_cgcs_score", "constraint_gate", "gated_macro_route", "decompression_recommended"]].head()

## Gate summary by macro route

In [ ]:
gate_summary = (
    work.groupby("macro_route")
    .agg(
        windows=("window_id", "count"),
        mean_cgcs=("macro_cgcs_score", "mean"),
        min_cgcs=("macro_cgcs_score", "min"),
        accepted_rate=("constraint_gate", lambda s: (s == "accepted").mean()),
        watch_rate=("constraint_gate", lambda s: (s == "watch").mean()),
        fallback_rate=("constraint_gate", lambda s: (s == "fallback").mean()),
        decompression_rate=("decompression_recommended", "mean"),
        mean_residual=("macro_compression_residual", "mean"),
        mean_stability=("compressed_route_stability", "mean"),
        mean_pressure=("macro_switch_rate", "mean"),
    )
    .reset_index()
    .sort_values("mean_cgcs", ascending=False)
)

gate_summary

## Compare raw macro route vs gated macro route

In [ ]:
work["raw_macro_changed"] = work["macro_route"].ne(work["macro_route"].shift(1)).fillna(False)
work["gated_macro_changed"] = work["gated_macro_route"].ne(work["gated_macro_route"].shift(1)).fillna(False)

roll = 15
work["raw_macro_switch_rate"] = work["raw_macro_changed"].rolling(roll, min_periods=1).mean()
work["gated_macro_switch_rate"] = work["gated_macro_changed"].rolling(roll, min_periods=1).mean()

work["gated_stability_score"] = (
    1.0
    - 0.60 * work["gated_macro_switch_rate"]
    - 0.25 * (work["constraint_gate"] == "fallback").astype(float)
    - 0.15 * work["decompression_recommended"].astype(float)
).clip(0, 1)

comparison = {
    "mean_raw_macro_switch_rate": float(work["raw_macro_switch_rate"].mean()),
    "mean_gated_macro_switch_rate": float(work["gated_macro_switch_rate"].mean()),
    "mean_macro_cgcs_score": float(work["macro_cgcs_score"].mean()),
    "accepted_windows": int((work["constraint_gate"] == "accepted").sum()),
    "watch_windows": int((work["constraint_gate"] == "watch").sum()),
    "fallback_windows": int((work["constraint_gate"] == "fallback").sum()),
    "decompression_recommended_windows": int(work["decompression_recommended"].sum()),
    "mean_gated_stability_score": float(work["gated_stability_score"].mean()),
}

comparison

## Gate transition matrix

In [ ]:
gate_labels = ["accepted", "watch", "fallback"]
gate_counts = pd.DataFrame(0, index=gate_labels, columns=gate_labels, dtype=float)

for a, b in zip(work["constraint_gate"].iloc[:-1], work["constraint_gate"].iloc[1:]):
    gate_counts.loc[a, b] += 1

gate_transition_probs = gate_counts.div(gate_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
gate_transition_probs

## Export constraint-gating outputs

In [ ]:
csv_path = RESULTS_DIR / "notebook20_constraint_gated_macro_routing.csv"
json_path = RESULTS_DIR / "notebook20_constraint_gated_macro_routing.json"
gate_summary_csv_path = RESULTS_DIR / "notebook20_gate_summary_by_macro_route.csv"
gate_transition_csv_path = RESULTS_DIR / "notebook20_gate_transition_matrix.csv"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)
gate_summary.to_csv(gate_summary_csv_path, index=False)
gate_transition_probs.to_csv(gate_transition_csv_path)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", gate_summary_csv_path)
print("Saved:", gate_transition_csv_path)

## Figure 1 — CGCS score timeline

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook20_macro_cgcs_score_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["macro_cgcs_score"], label="macro CGCS score")
plt.axhline(accept_threshold, linestyle="--", label="accept threshold")
plt.axhline(watch_threshold, linestyle="--", label="watch threshold")
fallback_rows = work[work["constraint_gate"] == "fallback"]
plt.scatter(fallback_rows["window_id"], fallback_rows["macro_cgcs_score"], s=25, label="fallback")
plt.xlabel("Window")
plt.ylabel("CGCS-style score")
plt.title("Constraint-Gated Macro Routing: CGCS Score Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Gate state timeline

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook20_constraint_gate_timeline.png"

labels = ["fallback", "watch", "accepted"]
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], work["constraint_gate"].map(lab_to_id), where="mid")
plt.yticks(list(lab_to_id.values()), labels)
plt.xlabel("Window")
plt.ylabel("Constraint gate")
plt.title("Constraint-Gated Macro Routing: Gate Timeline")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Gate summary by macro route

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook20_gate_summary_by_macro_route.png"

plot_df = gate_summary.sort_values("mean_cgcs")
plt.figure(figsize=(10, 5))
plt.bar(plot_df["macro_route"], plot_df["mean_cgcs"])
plt.axhline(accept_threshold, linestyle="--")
plt.axhline(watch_threshold, linestyle="--")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean CGCS-style score")
plt.title("Constraint-Gated Macro Routing: Mean Score by Macro Route")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Component matrix by macro route

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook20_component_matrix_by_macro_route.png"

component_summary = (
    work.groupby("macro_route")[[
        "coherence_component",
        "pressure_component",
        "reconstruction_component",
        "memory_component",
        "macro_cgcs_score",
    ]]
    .mean()
    .sort_values("macro_cgcs_score", ascending=False)
)

plt.figure(figsize=(9, 5))
plt.imshow(component_summary.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(component_summary.columns)), component_summary.columns, rotation=45, ha="right")
plt.yticks(range(len(component_summary.index)), component_summary.index)
plt.colorbar(label="Mean component score")
plt.title("Constraint-Gated Macro Routing: Component Matrix")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Raw vs gated macro switch rates

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook20_raw_vs_gated_switch_rates.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["raw_macro_switch_rate"], label="raw macro switch rate")
plt.plot(work["window_id"], work["gated_macro_switch_rate"], label="gated macro switch rate")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.title("Constraint-Gated Macro Routing: Raw vs Gated Switch Rates")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Decompression recommendation timeline

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook20_decompression_recommendation_timeline.png"

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], work["decompression_recommended"].astype(int), where="mid")
plt.yticks([0, 1], ["retain compressed", "decompress"])
plt.xlabel("Window")
plt.ylabel("Recommendation")
plt.title("Constraint-Gated Macro Routing: Decompression Recommendation")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Figure 7 — Gate transition matrix

In [ ]:
fig_path_7 = FIGURES_DIR / "notebook20_gate_transition_matrix.png"

plt.figure(figsize=(6, 5))
plt.imshow(gate_transition_probs.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(gate_labels)), gate_labels, rotation=45, ha="right")
plt.yticks(range(len(gate_labels)), gate_labels)
plt.colorbar(label="Transition probability")
plt.xlabel("Next gate")
plt.ylabel("Current gate")
plt.title("Constraint-Gated Macro Routing: Gate Transition Matrix")
plt.tight_layout()
plt.savefig(fig_path_7, dpi=160)
plt.show()

print("Saved:", fig_path_7)

## Figure 8 — Gated stability score

In [ ]:
fig_path_8 = FIGURES_DIR / "notebook20_gated_stability_score.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["gated_stability_score"])
plt.xlabel("Window")
plt.ylabel("Gated stability score")
plt.title("Constraint-Gated Macro Routing: Gated Stability Score")
plt.tight_layout()
plt.savefig(fig_path_8, dpi=160)
plt.show()

print("Saved:", fig_path_8)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_20_constraint_gated_macro_routing.md"

summary = {
    "windows": int(len(work)),
    "macro_route_count": int(work["macro_route"].nunique()),
    "mean_macro_cgcs_score": float(work["macro_cgcs_score"].mean()),
    "accepted_windows": int((work["constraint_gate"] == "accepted").sum()),
    "watch_windows": int((work["constraint_gate"] == "watch").sum()),
    "fallback_windows": int((work["constraint_gate"] == "fallback").sum()),
    "decompression_recommended_windows": int(work["decompression_recommended"].sum()),
    "mean_raw_macro_switch_rate": float(work["raw_macro_switch_rate"].mean()),
    "mean_gated_macro_switch_rate": float(work["gated_macro_switch_rate"].mean()),
    "mean_gated_stability_score": float(work["gated_stability_score"].mean()),
}

lines = [
    "# Report 20 — Constraint-Gated Macro Routing",
    "",
    "This report adds a CGCS-style constraint gate over compressed macro routes.",
    "",
    "Constraint view:",
    "> compressed memory is useful only when macro routes remain coherent, stable, and recoverable under observed pressure.",
    "",
    "## Generated outputs",
    "",
    f"- Constraint-gated routing CSV: `{csv_path}`",
    f"- Constraint-gated routing JSON: `{json_path}`",
    f"- Gate summary CSV: `{gate_summary_csv_path}`",
    f"- Gate transition matrix CSV: `{gate_transition_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    f"- Figure: `{fig_path_7}`",
    f"- Figure: `{fig_path_8}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Gate summary by macro route",
    "",
    gate_summary.to_markdown(index=False),
    "",
    "## Gate transition probabilities",
    "",
    gate_transition_probs.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Accepted macro routes are compressed states that remain coherent under route pressure.",
    "- Watch routes are retained but monitored for instability.",
    "- Fallback routes are blocked or rerouted because compression is not currently trustworthy.",
    "- Decompression recommendations identify when recursive compression should temporarily expand.",
    "",
    "## Next step",
    "",
    "Notebook 21 can build graph-based route topology over accepted, watch, and fallback macro states.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook20_constraint_gated_macro_routing_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook20_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_20_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))